In [47]:
#cargamos las tres novelas y aplicamos segmentacion y tokenizacion

from pathlib import Path
import re
import pandas as pd
import random

CSV_PATH = Path("gutenberg_novels_dataset.csv")

df = pd.read_csv(CSV_PATH)

#revisamos los libros disponibles
print("Libros encontrados en el dataset")
print(df[["title", "author"]])

#seleccionamos las tres novelas del laboratorio
novelas = df[df["title"].isin([
    "Pride and Prejudice",
    "Frankenstein",
    "Dracula"
])].copy()

if len(novelas) != 3:
    raise ValueError("No se encontraron exactamente las tres novelas esperadas")

#tokenizamos cada novela por separado
patron_palabras = re.compile(
    r"[A-Za-zÁÉÍÓÚÜÑáéíóúüñ]+(?:[-'][A-Za-zÁÉÍÓÚÜÑáéíóúüñ]+)*|\d+(?:[.,]\d+)*",
    re.UNICODE
)

#procesamos cada novela y guardamos los resultados en un diccionario
novelas_procesadas = {}

for _, fila in novelas.iterrows():
    texto = str(fila["text"])
    texto = re.sub(r"\s+", " ", texto).strip()

    #hacemos la segmentacion en oraciones
    oraciones_raw = re.split(r"(?<=[.!?])\s+", texto)

    #tokenizamos cada oracion sin quitar stopwords ni lematizar
    oraciones_tokenizadas = [
        patron_palabras.findall(oracion)
        for oracion in oraciones_raw
    ]

    oraciones_tokenizadas = [
        oracion
        for oracion in oraciones_tokenizadas
        if len(oracion) > 0
    ]

    novelas_procesadas[fila["title"]] = {
        "autor": fila["author"],
        "texto": texto,
        "oraciones": oraciones_tokenizadas
    }

    print(f"\nLibro {fila['title']}")
    print(f"Autor {fila['author']}")
    print(f"Total de oraciones {len(oraciones_tokenizadas):,}")
    print(f"Total de tokens {sum(len(oracion) for oracion in oraciones_tokenizadas):,}")

Libros encontrados en el dataset
                 title        author
0  Pride and Prejudice   Jane Austen
1         Frankenstein  Mary Shelley
2              Dracula   Bram Stoker

Libro Pride and Prejudice
Autor Jane Austen
Total de oraciones 5,943
Total de tokens 128,366

Libro Frankenstein
Autor Mary Shelley
Total de oraciones 3,122
Total de tokens 75,303

Libro Dracula
Autor Bram Stoker
Total de oraciones 7,839
Total de tokens 162,771


In [48]:
#seleccionamos una muestra aleatoria de 100 oraciones por novela

SEED = 42
random.seed(SEED)

TAMANO_MUESTRA = 100

for titulo, datos in novelas_procesadas.items():
    if len(datos["oraciones"]) < TAMANO_MUESTRA:
        raise ValueError(f"{titulo} no tiene suficientes oraciones para crear la muestra")

    datos["muestra"] = random.sample(
        datos["oraciones"],
        TAMANO_MUESTRA
    )

    datos["tokens_muestra"] = sum(
        len(oracion)
        for oracion in datos["muestra"]
    )
    
    #mostramos un resumen de la muestra

    print(f"\nMuestra de {titulo}")
    print(f"Oraciones seleccionadas {len(datos['muestra']):,}")
    print(f"Tokens en la muestra {datos['tokens_muestra']:,}")
    print("Ejemplo de oracion de la muestra")
    print(datos["muestra"][0])


Muestra de Pride and Prejudice
Oraciones seleccionadas 100
Tokens en la muestra 1,980
Ejemplo de oracion de la muestra
['After', 'this', 'day', 'Jane', 'said', 'no', 'more', 'of', 'her', 'indifference']

Muestra de Frankenstein
Oraciones seleccionadas 100
Tokens en la muestra 2,309
Ejemplo de oracion de la muestra
['I', 'was', 'partly', 'urged', 'by', 'curiosity', 'and', 'compassion', 'confirmed', 'my', 'resolution']

Muestra de Dracula
Oraciones seleccionadas 100
Tokens en la muestra 1,731
Ejemplo de oracion de la muestra
['I', 'shall', 'wire', 'to', 'my', 'people', 'to', 'have', 'horses', 'and', 'carriages', 'where', 'they', 'will', 'be', 'most', 'convenient', 'Look', 'here', 'old', 'fellow', 'said', 'Morris', 'it', 'is', 'a', 'capital', 'idea', 'to', 'have', 'all', 'ready', 'in', 'case', 'we', 'want', 'to', 'go', 'horsebacking', 'but', 'don', 't', 'you', 'think', 'that', 'one', 'of', 'your', 'snappy', 'carriages', 'with', 'its', 'heraldic', 'adornments', 'in', 'a', 'byway', 'of', '

In [49]:
#construimos la tabla comparativa de libros completos y muestras

tabla_muestras = []

for titulo, datos in novelas_procesadas.items():
    tabla_muestras.append({
        "libro": titulo,
        "oraciones_totales": len(datos["oraciones"]),
        "tokens_totales": sum(len(oracion) for oracion in datos["oraciones"]),
        "oraciones_muestra": len(datos["muestra"]),
        "tokens_muestra": datos["tokens_muestra"]
    })

tabla_muestras = pd.DataFrame(tabla_muestras)

display(tabla_muestras)

print("\nResumen de la preparacion de la muestra")

for _, fila in tabla_muestras.iterrows():
    print(
        f"{fila['libro']} | "
        f"oraciones totales {fila['oraciones_totales']:,} | "
        f"tokens totales {fila['tokens_totales']:,} | "
        f"oraciones muestra {fila['oraciones_muestra']:,} | "
        f"tokens muestra {fila['tokens_muestra']:,}"
    )

,libro,oraciones_totales,tokens_totales,oraciones_muestra,tokens_muestra
0,Pride and Prejudice,5943,128366,100,1980
1,Frankenstein,3122,75303,100,2309
2,Dracula,7839,162771,100,1731



Resumen de la preparacion de la muestra
Pride and Prejudice | oraciones totales 5,943 | tokens totales 128,366 | oraciones muestra 100 | tokens muestra 1,980
Frankenstein | oraciones totales 3,122 | tokens totales 75,303 | oraciones muestra 100 | tokens muestra 2,309
Dracula | oraciones totales 7,839 | tokens totales 162,771 | oraciones muestra 100 | tokens muestra 1,731


In [50]:
#instalamos el modelo de spacy para etiquetado gramatical

!python -m spacy download en_core_web_sm

     ---------------------------------------- 0.0/12.8 MB ? eta -:--:--
     ---------------------------------------- 0.0/12.8 MB ? eta -:--:--
     ---- ----------------------------------- 1.3/12.8 MB 7.1 MB/s eta 0:00:02
     -------------- ------------------------- 4.7/12.8 MB 11.9 MB/s eta 0:00:01
     ------------------------ --------------- 7.9/12.8 MB 13.2 MB/s eta 0:00:01
     ----------------------------------- --- 11.5/12.8 MB 14.2 MB/s eta 0:00:01
     ---------------------------------------- 12.8/12.8 MB 14.1 MB/s  0:00:01
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')



[notice] A new release of pip is available: 26.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [51]:
#etiquetamos cada token de las muestras usando spacy

import spacy

nlp = spacy.load("en_core_web_sm")

for titulo, datos in novelas_procesadas.items():
    muestras_pos = []

    for oracion in datos["muestra"]:
        texto_oracion = " ".join(oracion)
        doc = nlp(texto_oracion)

        etiquetas_oracion = [
            (token.text, token.pos_)
            for token in doc
        ]

        muestras_pos.append(etiquetas_oracion)

    datos["muestra_pos"] = muestras_pos

    print(f"\nLibro {titulo}")
    print("Ejemplo de oracion etiquetada")
    print(datos["muestra_pos"][0])


Libro Pride and Prejudice
Ejemplo de oracion etiquetada
[('After', 'ADP'), ('this', 'DET'), ('day', 'NOUN'), ('Jane', 'PROPN'), ('said', 'VERB'), ('no', 'DET'), ('more', 'ADJ'), ('of', 'ADP'), ('her', 'PRON'), ('indifference', 'NOUN')]

Libro Frankenstein
Ejemplo de oracion etiquetada
[('I', 'PRON'), ('was', 'AUX'), ('partly', 'ADV'), ('urged', 'VERB'), ('by', 'ADP'), ('curiosity', 'NOUN'), ('and', 'CCONJ'), ('compassion', 'NOUN'), ('confirmed', 'VERB'), ('my', 'PRON'), ('resolution', 'NOUN')]

Libro Dracula
Ejemplo de oracion etiquetada
[('I', 'PRON'), ('shall', 'AUX'), ('wire', 'VERB'), ('to', 'ADP'), ('my', 'PRON'), ('people', 'NOUN'), ('to', 'PART'), ('have', 'VERB'), ('horses', 'NOUN'), ('and', 'CCONJ'), ('carriages', 'NOUN'), ('where', 'SCONJ'), ('they', 'PRON'), ('will', 'AUX'), ('be', 'AUX'), ('most', 'ADV'), ('convenient', 'ADJ'), ('Look', 'PROPN'), ('here', 'ADV'), ('old', 'ADJ'), ('fellow', 'NOUN'), ('said', 'VERB'), ('Morris', 'PROPN'), ('it', 'PRON'), ('is', 'AUX'), ('a',

In [52]:
#calculamos la distribucion de categorias gramaticales de cada libro

from collections import Counter

distribuciones_pos = {}

for titulo, datos in novelas_procesadas.items():
    conteo_pos = Counter()

    for oracion in datos["muestra_pos"]:
        for palabra, etiqueta in oracion:
            conteo_pos[etiqueta] += 1

    distribuciones_pos[titulo] = conteo_pos

#construimos una tabla comparativa entre los tres libros

categorias_pos = sorted(
    set().union(
        *[set(conteo.keys()) for conteo in distribuciones_pos.values()]
    )
)

tabla_pos = pd.DataFrame(index=categorias_pos)

for titulo, conteo in distribuciones_pos.items():
    tabla_pos[titulo] = [
        conteo.get(categoria, 0)
        for categoria in categorias_pos
    ]

tabla_pos.index.name = "POS"
tabla_pos = tabla_pos.fillna(0).astype(int)

display(tabla_pos)

,Pride and Prejudice,Frankenstein,Dracula
POS,,,
ADJ,122,160,106
ADP,213,289,186
ADV,148,98,87
AUX,181,160,123
CCONJ,77,106,85
DET,153,233,141
INTJ,11,4,4
NOUN,262,452,254
NUM,10,12,7


In [53]:
#buscamos palabras que recibieron distintas etiquetas en diferentes contextos

ocurrencias_palabras = {}

for titulo, datos in novelas_procesadas.items():
    for oracion in datos["muestra_pos"]:
        for i, (palabra, etiqueta) in enumerate(oracion):
            palabra_clave = palabra.lower()

            if palabra_clave not in ocurrencias_palabras:
                ocurrencias_palabras[palabra_clave] = []

            inicio = max(0, i - 3)
            fin = min(len(oracion), i + 4)

            contexto = " ".join(
                token
                for token, _ in oracion[inicio:fin]
            )

            ocurrencias_palabras[palabra_clave].append({
                "libro": titulo,
                "palabra": palabra,
                "etiqueta": etiqueta,
                "contexto": contexto
            })

#seleccionamos palabras con mas de una etiqueta distinta

ambiguedades_pos = []

for palabra, ocurrencias in ocurrencias_palabras.items():
    etiquetas_distintas = set(
        ocurrencia["etiqueta"]
        for ocurrencia in ocurrencias
    )

    if len(etiquetas_distintas) > 1:
        ambiguedades_pos.append(
            (palabra, ocurrencias)
        )

#mostramos al menos tres palabras con ambiguedad de etiquetas

palabras_mostradas = 0

for palabra, ocurrencias in ambiguedades_pos:
    etiquetas_mostradas = set()

    print(f"\nPalabra {palabra}")

    for ocurrencia in ocurrencias:
        if ocurrencia["etiqueta"] not in etiquetas_mostradas:
            print(f"Libro {ocurrencia['libro']}")
            print(f"Etiqueta {ocurrencia['etiqueta']}")
            print(f"Contexto {ocurrencia['contexto']}")
            print()

            etiquetas_mostradas.add(
                ocurrencia["etiqueta"]
            )

    if len(etiquetas_mostradas) > 1:
        palabras_mostradas += 1

    if palabras_mostradas == 3:
        break


Palabra after
Libro Pride and Prejudice
Etiqueta ADP
Contexto After this day Jane

Libro Pride and Prejudice
Etiqueta SCONJ
Contexto Soon after you left me


Palabra this
Libro Pride and Prejudice
Etiqueta DET
Contexto After this day Jane said

Libro Pride and Prejudice
Etiqueta PRON
Contexto This was one point


Palabra no
Libro Pride and Prejudice
Etiqueta DET
Contexto day Jane said no more of her

Libro Pride and Prejudice
Etiqueta PRON
Contexto but could do no more

Libro Frankenstein
Etiqueta ADV
Contexto human frame could no longer support the



Algunas asignaciones parecen incorrectas, especialmente en palabras que cambian de función según el contexto, como “no” o “after”, y en casos como “Look” en Dracula, que fue etiquetado como PROPN aunque funciona como verbo. De los tres, Dracula muestra errores más visibles, probablemente por sus formas de diálogo, vocabulario y construcciones menos comunes para un tagger entrenado con inglés contemporáneo.

In [54]:
#extraemos las etiquetas universal y penn treebank de 10 tokens

titulo_libro = "Pride and Prejudice"
oracion_ejemplo = novelas_procesadas[titulo_libro]["muestra"][0]

doc = nlp(" ".join(oracion_ejemplo))

resultado_tags = []

for token in doc[:10]:
    resultado_tags.append({
        "token": token.text,
        "POS_universal": token.pos_,
        "tag_Penn_Treebank": token.tag_
    })

tabla_tags = pd.DataFrame(resultado_tags)

display(tabla_tags)

,token,POS_universal,tag_Penn_Treebank
0,After,ADP,IN
1,this,DET,DT
2,day,NOUN,NN
3,Jane,PROPN,NNP
4,said,VERB,VBD
5,no,DET,DT
6,more,ADJ,JJR
7,of,ADP,IN
8,her,PRON,PRP$
9,indifference,NOUN,NN


El tagset universal muestra la categoría gramatical general de cada palabra, mientras que Penn Treebank aporta un nivel de detalle mayor. Por ejemplo, “said” aparece como VERB, pero VBD indica específicamente que es un verbo en pasado; “more” es ADJ, pero JJR indica que es un adjetivo comparativo, y “Jane” es PROPN, mientras que NNP especifica que es un nombre propio en singular. En otros casos, como “day” (NOUN → NN) o “this” (DET → DT), la etiqueta fina también precisa el tipo concreto de palabra. Por eso, Penn Treebank permite representar diferencias gramaticales que el tagset universal agrupa en una sola categoría.

In [55]:
#seleccionamos cinco oraciones especificas del fragmento de Dracula
#y mostramos sus etiquetas Penn Treebank con spacy

oraciones_penn = [
    "One of the letters was directed to Samuel F. Billington, No. 7, The Crescent, Whitby, another to Herr Leutner, Varna; the third was to Coutts & Co., London, and the fourth to Herren Klopstock & Billreuth, bankers, Buda-Pesth.",
    
    "The second and fourth were unsealed.",
    
    "I was just about to look at them when I saw the door-handle move.",
    
    "I sank back in my seat, having just had time to replace the letters as they had been and to resume my book before the Count, holding still another letter in his hand, entered the room.",
    
    "He took up the letters on the table and stamped them carefully, and then turning to me, said:--"
]

for numero, texto_oracion in enumerate(oraciones_penn, start=1):
    doc = nlp(texto_oracion)

    print(f"\nOracion {numero}")
    print(texto_oracion)

    for token in doc:
        print(f"{token.text} -> {token.tag_}")


Oracion 1
One of the letters was directed to Samuel F. Billington, No. 7, The Crescent, Whitby, another to Herr Leutner, Varna; the third was to Coutts & Co., London, and the fourth to Herren Klopstock & Billreuth, bankers, Buda-Pesth.
One -> CD
of -> IN
the -> DT
letters -> NNS
was -> VBD
directed -> VBN
to -> IN
Samuel -> NNP
F. -> NNP
Billington -> NNP
, -> ,
No -> NNP
. -> NN
7 -> CD
, -> ,
The -> DT
Crescent -> NNP
, -> ,
Whitby -> NNP
, -> ,
another -> DT
to -> IN
Herr -> NNP
Leutner -> NNP
, -> ,
Varna -> NNP
; -> :
the -> DT
third -> JJ
was -> VBD
to -> IN
Coutts -> NNP
& -> CC
Co. -> NNP
, -> ,
London -> NNP
, -> ,
and -> CC
the -> DT
fourth -> JJ
to -> IN
Herren -> NNP
Klopstock -> NNP
& -> CC
Billreuth -> NNP
, -> ,
bankers -> NNS
, -> ,
Buda -> NNP
- -> HYPH
Pesth -> NNP
. -> .

Oracion 2
The second and fourth were unsealed.
The -> DT
second -> JJ
and -> CC
fourth -> JJ
were -> VBD
unsealed -> VBN
. -> .

Oracion 3
I was just about to look at them when I saw the door-handl

En total, se compararon 136 tokens entre el etiquetado manual y el etiquetado automático. En total hubieron 132 coincidencias de 136 tokens, con una diferencia de 4 tokens. La mayoria de etiquestas coincidieron con el taagger automatico, los descauerdos principales probablemente se dieron en casos donde la tokenizacion o el contexto gramatical podrian confundir al modelo, especialmente con puntuacion pegada a palabras como "said", y con palabras ambiguas como "about" o "move".

In [56]:
#extraemos los primeros tres capitulos y aplicamos ner

import re
import spacy

def convertir_numero_capitulo(numero):
    numero = numero.upper()

    if numero.isdigit():
        return int(numero)

    valores = {
        "I": 1,
        "V": 5,
        "X": 10,
        "L": 50,
        "C": 100,
        "D": 500,
        "M": 1000
    }

    total = 0
    anterior = 0

    for letra in reversed(numero):
        valor = valores[letra]

        if valor < anterior:
            total -= valor
        else:
            total += valor

        anterior = valor

    return total


def extraer_tres_capitulos(texto):
    #buscamos encabezados de capitulos en todo el texto
    patron_capitulo = re.compile(
        r"(?i)\bchapter\s+([ivxlcdm]+|\d+)\b"
    )

    coincidencias = list(patron_capitulo.finditer(texto))

    if len(coincidencias) < 4:
        raise ValueError(
            f"No se encontraron suficientes encabezados de capitulos. "
            f"Se encontraron {len(coincidencias)}"
        )

    #guardamos el numero de cada capitulo encontrado
    candidatos = []

    for coincidencia in coincidencias:
        numero = convertir_numero_capitulo(coincidencia.group(1))

        candidatos.append({
            "numero": numero,
            "inicio": coincidencia.start()
        })

    #buscamos una secuencia real de capitulos 1, 2 y 3
    #evitando secuencias demasiado juntas como las de una tabla de contenido
    for i in range(len(candidatos) - 3):
        actual = candidatos[i]
        siguiente = candidatos[i + 1]
        tercero = candidatos[i + 2]
        cuarto = candidatos[i + 3]

        if (
            actual["numero"] == 1
            and siguiente["numero"] == 2
            and tercero["numero"] == 3
            and siguiente["inicio"] - actual["inicio"] > 300
            and tercero["inicio"] - siguiente["inicio"] > 300
        ):
            inicio = actual["inicio"]
            fin = cuarto["inicio"]

            return texto[inicio:fin]

    raise ValueError("No se encontro una secuencia valida de los primeros tres capitulos")


#usamos el texto original del dataset para conservar la estructura de la novela

for titulo, datos in novelas_procesadas.items():
    fila_libro = novelas[novelas["title"] == titulo].iloc[0]
    texto_original = str(fila_libro["text"])

    texto_capitulos = extraer_tres_capitulos(texto_original)
    doc_ner = nlp(texto_capitulos)

    datos["texto_ner"] = texto_capitulos
    datos["doc_ner"] = doc_ner

    print(f"\nLibro {titulo}")
    print("Primeros tres capitulos procesados")
    print(f"Total de tokens {len(doc_ner):,}")
    print(f"Total de entidades detectadas {len(doc_ner.ents):,}")


Libro Pride and Prejudice
Primeros tres capitulos procesados
Total de tokens 4,575
Total de entidades detectadas 190

Libro Frankenstein
Primeros tres capitulos procesados
Total de tokens 8,033
Total de entidades detectadas 155

Libro Dracula
Primeros tres capitulos procesados
Total de tokens 21,021
Total de entidades detectadas 388


In [57]:
#verificamos que el texto extraido corresponda a los primeros tres capitulos

for titulo, datos in novelas_procesadas.items():
    texto = datos["texto_ner"]

    encabezados = re.findall(
        r"(?i)\bchapter\s+(?:[ivxlcdm]+|\d+)\b",
        texto
    )

    print(f"\nLibro {titulo}")
    print("Encabezados encontrados")
    print(encabezados[:5])
    print(f"Cantidad de encabezados encontrados {len(encabezados)}")
    print(f"Total de tokens {len(datos['doc_ner']):,}")


Libro Pride and Prejudice
Encabezados encontrados
['Chapter I', 'CHAPTER II', 'CHAPTER III']
Cantidad de encabezados encontrados 3
Total de tokens 4,575

Libro Frankenstein
Encabezados encontrados
['Chapter 1', 'Chapter 2', 'Chapter 3']
Cantidad de encabezados encontrados 3
Total de tokens 8,033

Libro Dracula
Encabezados encontrados
['CHAPTER I', 'CHAPTER II', 'CHAPTER III']
Cantidad de encabezados encontrados 3
Total de tokens 21,021


In [58]:
#mostramos las entidades detectadas para revisar personajes y lugares

for titulo, datos in novelas_procesadas.items():
    personas = Counter(
        entidad.text
        for entidad in datos["doc_ner"].ents
        if entidad.label_ == "PERSON"
    )

    lugares = Counter(
        entidad.text
        for entidad in datos["doc_ner"].ents
        if entidad.label_ in {"GPE", "LOC"}
    )

    print(f"\nLibro {titulo}")

    print("\nPersonajes detectados")
    for entidad, frecuencia in personas.most_common(10):
        print(f"{entidad} -> {frecuencia}")

    print("\nLugares detectados")
    for entidad, frecuencia in lugares.most_common(10):
        print(f"{entidad} -> {frecuencia}")


Libro Pride and Prejudice

Personajes detectados
Bennet -> 24
Bingley -> 24
Lizzy -> 7
Long -> 6
Jane -> 6
Darcy -> 6
George Allen -> 5
Elizabeth -> 4
Mary -> 4
Lady Lucas -> 3

Lugares detectados
Netherfield -> 2
London -> 2
England -> 1
Lydia -> 1
Hertfordshire -> 1

Libro Frankenstein

Personajes detectados
Elizabeth -> 12
Clerval -> 5
Paracelsus -> 4
M. Waldman -> 4
Cornelius Agrippa -> 3
Agrippa -> 2
M. Krempe -> 2
Caroline -> 1
Elizabeth Lavenza -> 1
Alpine -> 1

Lugares detectados
Geneva -> 5
Beaufort -> 4
Italy -> 4
M. Krempe -> 3
Milan -> 2
Lucerne -> 1
Germany -> 1
France -> 1
Naples -> 1
the Lake of Como -> 1

Libro Dracula

Personajes detectados
Dracula -> 9
Mem -> 5
Szekelys -> 4
Turk -> 4
Hawkins -> 4
JONATHAN HARKER -> 3
Herr -> 3
Exeter -> 3
Danube -> 2
Count Dracula -> 2

Lugares detectados
London -> 16
Transylvania -> 6
Bukovina -> 6
England -> 6
South -> 3
West -> 2
East -> 2
Europe -> 2
Slovaks -> 2
Magyar -> 2


En general si coinciden, pero los resultados muestran algunas diferencias. En Pride and Prejudice aparecen personajes esperables como Bennet, Bingley, Jane y Darcy, y lugares como Netherfield y London. En Frankenstein aparecen Elizabeth, Clerval y Paracelsus, junto con lugares como Geneva, Italy y Milan. En Dracula aparecen Dracula, Hawkins y Jonathan Harker, y lugares como London, Transylvania, Bukovina y England. Por lo tanto, la mayoria de las entidades detectadas si corresponden con personajes y lugares reconocibles de cada novela, aunque tambien aparecen algunas entidades clasificadas de forma dudosa.

In [59]:
#calculamos la distribucion de entidades de cada libro

from collections import Counter

resultados_ner = []

for titulo, datos in novelas_procesadas.items():
    conteo_entidades = Counter(
        entidad.label_
        for entidad in datos["doc_ner"].ents
    )

    fila = {
        "libro": titulo,
        "entidades_totales": len(datos["doc_ner"].ents)
    }

    for etiqueta, cantidad in conteo_entidades.items():
        fila[etiqueta] = cantidad

    resultados_ner.append(fila)

#construimos una tabla comparativa de entidades

tabla_ner = pd.DataFrame(resultados_ner)
tabla_ner = tabla_ner.fillna(0)

#ordenamos las columnas para comparar los tres libros

columnas = ["libro", "entidades_totales"] + sorted(
    columna
    for columna in tabla_ner.columns
    if columna not in {"libro", "entidades_totales"}
)

tabla_ner = tabla_ner[columnas]

#convertimos las cantidades a enteros

columnas_entidades = [
    columna
    for columna in tabla_ner.columns
    if columna != "libro"
]

tabla_ner[columnas_entidades] = tabla_ner[columnas_entidades].astype(int)

display(tabla_ner)

,libro,entidades_totales,CARDINAL,DATE,EVENT,FAC,GPE,LANGUAGE,LAW,LOC,NORP,ORDINAL,ORG,PERSON,PRODUCT,QUANTITY,TIME,WORK_OF_ART
0,Pride and Prejudice,190,27,16,0,2,7,0,0,0,0,7,9,110,0,1,10,1
1,Frankenstein,155,15,31,0,1,26,0,3,2,8,8,9,41,0,1,8,2
2,Dracula,388,71,30,1,5,56,6,2,16,28,13,42,77,3,4,31,3


In [60]:
#calculamos la densidad de entidades por cada 100 tokens

densidad_entidades = []

for titulo, datos in novelas_procesadas.items():
    tokens_totales = len(datos["doc_ner"])
    entidades_totales = len(datos["doc_ner"].ents)

    densidad = (
        entidades_totales / tokens_totales
    ) * 100

    densidad_entidades.append({
        "libro": titulo,
        "tokens": tokens_totales,
        "entidades": entidades_totales,
        "entidades_por_100_tokens": densidad
    })

tabla_densidad = pd.DataFrame(densidad_entidades)

tabla_densidad["entidades_por_100_tokens"] = (
    tabla_densidad["entidades_por_100_tokens"].round(2)
)

display(tabla_densidad)

,libro,tokens,entidades,entidades_por_100_tokens
0,Pride and Prejudice,4575,190,4.15
1,Frankenstein,8033,155,1.93
2,Dracula,21021,388,1.85


Pride and Prejudice tiene la mayor densidad de entidades con 4.15 por cada 100 tokens, mientras que Frankenstein tiene 1.93 y Dracula 1.85. Tambien es el libro con mas entidades PERSON, con 110 frente a 41 en Frankenstein y 77 en Dracula. En esta muestra se observa una diferencia clara en la cantidad de nombres propios, aunque estos resultados por si solos no permiten afirmar que la diferencia se deba directamente al genero o estilo narrativo.

In [61]:
#aplicamos ner al fragmento seleccionado

fragmento = """
When the ladies returned to the drawing-room, there was little to be
done but to hear Lady Catherine talk, which she did without any
intermission till coffee came in, delivering her opinion on every
subject in so decisive a manner as proved that she was not used to have
her judgment controverted. She inquired into Charlotte’s domestic
concerns familiarly and minutely, and gave her a great deal of advice as
to the management of them all; told her how everything ought to be
regulated in so small a family as hers, and instructed her as to the
care of her cows and her poultry. Elizabeth found that nothing was
beneath this great lady’s attention which could furnish her with an
occasion for dictating to others. In the intervals of her discourse with
Mrs. Collins, she addressed a variety of questions to Maria and
Elizabeth, but especially to the latter, of whose connections she knew
the least, and who, she observed to Mrs. Collins, was a very genteel,
pretty kind of girl. She asked her at different times how many sisters
she had, whether they were older or younger than herself, whether any of
them were likely to be married, whether they were handsome, where they
had been educated, what carriage her father kept, and what had been her
mother’s maiden name? Elizabeth felt all the impertinence of her
questions, but answered them very composedly. Lady Catherine then
observed,
"""

#quitamos los saltos de linea antes de aplicar el modelo

fragmento = re.sub(r"\s+", " ", fragmento).strip()

doc_fragmento = nlp(fragmento)

print("Entidades detectadas por el modelo")

for entidad in doc_fragmento.ents:
    print(f"{entidad.text} -> {entidad.label_}")

Entidades detectadas por el modelo
Lady Catherine -> PERSON
Charlotte -> GPE
Elizabeth -> PERSON
Collins -> PERSON
Maria -> NORP
Elizabeth -> PERSON
Collins -> PERSON
Elizabeth -> PERSON
Lady Catherine -> PERSON


In [62]:
#construimos manualmente las etiquetas BIO corregidas

tokens = [token.text for token in doc_fragmento]

etiquetas_bio = ["O"] * len(tokens)

#entidades del fragmento que consideramos entidades PERSON

entidades_manuales = [
    ["Lady", "Catherine"],
    ["Charlotte"],
    ["Elizabeth"],
    ["Mrs.", "Collins"],
    ["Maria"],
    ["Elizabeth"],
    ["Mrs.", "Collins"],
    ["Elizabeth"],
    ["Lady", "Catherine"]
]

#asignamos B e I a cada entidad

for entidad in entidades_manuales:
    for i in range(len(tokens) - len(entidad) + 1):
        if tokens[i:i + len(entidad)] == entidad:
            if etiquetas_bio[i] == "O":
                etiquetas_bio[i] = "B-PERSON"

                for j in range(1, len(entidad)):
                    etiquetas_bio[i + j] = "I-PERSON"

#mostramos el resultado como tabla de dos filas

tabla_bio = pd.DataFrame(
    [tokens, etiquetas_bio],
    index=["token", "etiqueta_BIO"]
)

display(tabla_bio)

,0,1,2,3,4,5,6,7,8,9,...,260,261,262,263,264,265,266,267,268,269
token,When,the,ladies,returned,to,the,drawing,-,room,",",...,answered,them,very,composedly,.,Lady,Catherine,then,observed,","
etiqueta_BIO,O,O,O,O,O,O,O,O,O,O,...,O,O,O,O,O,B-PERSON,I-PERSON,O,O,O


**Pride and Prejudice by Jane Austen**

In [63]:
#anotacion manual de entidades verdaderas para Pride and Prejudice

anotaciones_manuales = [
    {
        "oracion": 1,
        "texto": "My dearest sister, now be, be serious.",
        "entidades": []
    },
    {
        "oracion": 2,
        "texto": "I want to talk very seriously.",
        "entidades": []
    },
    {
        "oracion": 3,
        "texto": "Let me know everything that I am to know without delay.",
        "entidades": []
    },
    {
        "oracion": 4,
        "texto": "Will you tell me how long you have loved him?",
        "entidades": []
    },
    {
        "oracion": 5,
        "texto": "It has been coming on so gradually, that I hardly know when it began; but I believe I must date it from my first seeing his beautiful grounds at Pemberley.",
        "entidades": [
            ("Pemberley", "GPE")
        ]
    },
    {
        "oracion": 6,
        "texto": "Another entreaty that she would be serious, however, produced the desired effect; and she soon satisfied Jane by her solemn assurances of attachment.",
        "entidades": [
            ("Jane", "PERSON")
        ]
    },
    {
        "oracion": 7,
        "texto": "When convinced on that article, Miss Bennet had nothing further to wish.",
        "entidades": [
            ("Miss Bennet", "PERSON")
        ]
    },
    {
        "oracion": 8,
        "texto": "Now I am quite happy, said she, for you will be as happy as myself.",
        "entidades": []
    },
    {
        "oracion": 9,
        "texto": "I always had a value for him.",
        "entidades": []
    },
    {
        "oracion": 10,
        "texto": "Were it for nothing but his love of you, I must always have esteemed him; but now, as Bingley's friend and your husband, there can be only Bingley and yourself more dear to me.",
        "entidades": [
            ("Bingley", "PERSON"),
            ("Bingley", "PERSON")
        ]
    },
    {
        "oracion": 11,
        "texto": "But, Lizzy, you have been very sly, very reserved with me.",
        "entidades": [
            ("Lizzy", "PERSON")
        ]
    },
    {
        "oracion": 12,
        "texto": "How little did you tell me of what passed at Pemberley and Lambton!",
        "entidades": [
            ("Pemberley", "GPE"),
            ("Lambton", "GPE")
        ]
    },
    {
        "oracion": 13,
        "texto": "I owe all that I know of it to another, not to you.",
        "entidades": []
    },
    {
        "oracion": 14,
        "texto": "Elizabeth told her the motives of her secrecy.",
        "entidades": [
            ("Elizabeth", "PERSON")
        ]
    },
    {
        "oracion": 15,
        "texto": "She had been unwilling to mention Bingley; and the unsettled state of her own feelings had made her equally avoid the name of his friend: but now she would no longer conceal from her his share in Lydia's marriage.",
        "entidades": [
            ("Bingley", "PERSON"),
            ("Lydia", "PERSON")
        ]
    },
    {
        "oracion": 16,
        "texto": "All was acknowledged, and half the night spent in conversation.",
        "entidades": []
    },
    {
        "oracion": 17,
        "texto": "Good gracious!",
        "entidades": []
    },
    {
        "oracion": 18,
        "texto": "Cried Mrs. Bennet, as she stood at a window the next morning, if that disagreeable Mr. Darcy is not coming here again with our dear Bingley!",
        "entidades": [
            ("Mrs. Bennet", "PERSON"),
            ("Mr. Darcy", "PERSON"),
            ("Bingley", "PERSON")
        ]
    },
    {
        "oracion": 19,
        "texto": "What can he mean by being so tiresome as to be always coming here?",
        "entidades": []
    },
    {
        "oracion": 20,
        "texto": "I had no notion but he would go a-shooting, or something or other, and not disturb us with his company.",
        "entidades": []
    }
]

#mostramos la anotacion manual

for anotacion in anotaciones_manuales:
    print(f"\nOracion {anotacion['oracion']}")
    print(anotacion["texto"])

    if anotacion["entidades"]:
        for span, tipo in anotacion["entidades"]:
            print(f"{span} -> {tipo}")
    else:
        print("Sin entidades")


Oracion 1
My dearest sister, now be, be serious.
Sin entidades

Oracion 2
I want to talk very seriously.
Sin entidades

Oracion 3
Let me know everything that I am to know without delay.
Sin entidades

Oracion 4
Will you tell me how long you have loved him?
Sin entidades

Oracion 5
It has been coming on so gradually, that I hardly know when it began; but I believe I must date it from my first seeing his beautiful grounds at Pemberley.
Pemberley -> GPE

Oracion 6
Another entreaty that she would be serious, however, produced the desired effect; and she soon satisfied Jane by her solemn assurances of attachment.
Jane -> PERSON

Oracion 7
When convinced on that article, Miss Bennet had nothing further to wish.
Miss Bennet -> PERSON

Oracion 8
Now I am quite happy, said she, for you will be as happy as myself.
Sin entidades

Oracion 9
I always had a value for him.
Sin entidades

Oracion 10
Were it for nothing but his love of you, I must always have esteemed him; but now, as Bingley's friend

**Frankenstein by Mary Shelley**

In [64]:
#anotacion manual de entidades verdaderas para Frankenstein

anotaciones_manuales_frankenstein = [
    {
        "oracion": 1,
        "texto": "A woman deposed that she lived near the beach and was standing at the door of her cottage, waiting for the return of the fishermen, about an hour before she heard of the discovery of the body, when she saw a boat with only one man in it push off from that part of the shore where the corpse was afterwards found.",
        "entidades": []
    },
    {
        "oracion": 2,
        "texto": "Another woman confirmed the account of the fishermen having brought the body into her house; it was not cold.",
        "entidades": []
    },
    {
        "oracion": 3,
        "texto": "They put it into a bed and rubbed it, and Daniel went to the town for an apothecary, but life was quite gone.",
        "entidades": [
            ("Daniel", "PERSON")
        ]
    },
    {
        "oracion": 4,
        "texto": "Several other men were examined concerning my landing, and they agreed that, with the strong north wind that had arisen during the night, it was very probable that I had beaten about for many hours and had been obliged to return nearly to the same spot from which I had departed.",
        "entidades": []
    },
    {
        "oracion": 5,
        "texto": "Besides, they observed that it appeared that I had brought the body from another place, and it was likely that as I did not appear to know the shore, I might have put into the harbour ignorant of the distance of the town of —— from the place where I had deposited the corpse.",
        "entidades": []
    },
    {
        "oracion": 6,
        "texto": "Mr. Kirwin, on hearing this evidence, desired that I should be taken into the room where the body lay for interment, that it might be observed what effect the sight of it would produce upon me.",
        "entidades": [
            ("Mr. Kirwin", "PERSON")
        ]
    },
    {
        "oracion": 7,
        "texto": "This idea was probably suggested by the extreme agitation I had exhibited when the mode of the murder had been described.",
        "entidades": []
    },
    {
        "oracion": 8,
        "texto": "I was accordingly conducted, by the magistrate and several other persons, to the inn.",
        "entidades": []
    },
    {
        "oracion": 9,
        "texto": "I could not help being struck by the strange coincidences that had taken place during this eventful night; but, knowing that I had been conversing with several persons in the island I had inhabited about the time that the body had been found, I was perfectly tranquil as to the consequences of the affair.",
        "entidades": []
    },
    {
        "oracion": 10,
        "texto": "The examination, the presence of the magistrate and witnesses, passed like a dream from my memory when I saw the lifeless form of Henry Clerval stretched before me.",
        "entidades": [
            ("Henry Clerval", "PERSON")
        ]
    },
    {
        "oracion": 11,
        "texto": "I gasped for breath, and throwing myself on the body, I exclaimed, “Have my murderous machinations deprived you also, my dearest Henry, of life?",
        "entidades": [
            ("Henry", "PERSON")
        ]
    },
    {
        "oracion": 12,
        "texto": "Two I have already destroyed; other victims await their destiny; but you, Clerval, my friend, my benefactor—",
        "entidades": [
            ("Clerval", "PERSON")
        ]
    },
    {
        "oracion": 13,
        "texto": "A fever succeeded to this.",
        "entidades": []
    },
    {
        "oracion": 14,
        "texto": "I called myself the murderer of William, of Justine, and of Clerval.",
        "entidades": [
            ("William", "PERSON"),
            ("Justine", "PERSON"),
            ("Clerval", "PERSON")
        ]
    },
    {
        "oracion": 15,
        "texto": "Fortunately, as I spoke my native language, Mr. Kirwin alone understood me; but my gestures and bitter cries were sufficient to affright the other witnesses.",
        "entidades": [
            ("Mr. Kirwin", "PERSON")
        ]
    },
    {
        "oracion": 16,
        "texto": "Why did I not die?",
        "entidades": []
    },
    {
        "oracion": 17,
        "texto": "But I was doomed to live and in two months found myself as awaking from a dream, in a prison, stretched on a wretched bed, surrounded by gaolers, turnkeys, bolts, and all the miserable apparatus of a dungeon.",
        "entidades": []
    },
    {
        "oracion": 18,
        "texto": "I replied in the same language, with a feeble voice, “I believe I am; but if it be all true, if indeed I did not dream, I am sorry that I am still alive to feel this misery and horror.”",
        "entidades": []
    },
    {
        "oracion": 19,
        "texto": "This sound disturbed an old woman who was sleeping in a chair beside me.",
        "entidades": []
    },
    {
        "oracion": 20,
        "texto": "These were my first reflections, but I soon learned that Mr. Kirwin had shown me extreme kindness.",
        "entidades": [
            ("Mr. Kirwin", "PERSON")
        ]
    }

]

#mostramos las entidades anotadas manualmente

for anotacion in anotaciones_manuales_frankenstein:
    print(f"\nOracion {anotacion['oracion']}")
    print(anotacion["texto"])

    if anotacion["entidades"]:
        for span, tipo in anotacion["entidades"]:
            print(f"{span} -> {tipo}")
    else:
        print("Sin entidades")


Oracion 1
A woman deposed that she lived near the beach and was standing at the door of her cottage, waiting for the return of the fishermen, about an hour before she heard of the discovery of the body, when she saw a boat with only one man in it push off from that part of the shore where the corpse was afterwards found.
Sin entidades

Oracion 2
Another woman confirmed the account of the fishermen having brought the body into her house; it was not cold.
Sin entidades

Oracion 3
They put it into a bed and rubbed it, and Daniel went to the town for an apothecary, but life was quite gone.
Daniel -> PERSON

Oracion 4
Several other men were examined concerning my landing, and they agreed that, with the strong north wind that had arisen during the night, it was very probable that I had beaten about for many hours and had been obliged to return nearly to the same spot from which I had departed.
Sin entidades

Oracion 5
Besides, they observed that it appeared that I had brought the body from 

**Dracula by Bram Stoker**

In [65]:
#anotacion manual de entidades verdaderas para Dracula

anotaciones_manuales_dracula = [
    {
        "oracion": 1,
        "texto": "I am too agitated to sleep.",
        "entidades": []
    },
    {
        "oracion": 2,
        "texto": "We have had such an adventure, such an agonising experience.",
        "entidades": []
    },
    {
        "oracion": 3,
        "texto": "I fell asleep as soon as I had closed my diary.",
        "entidades": []
    },
    {
        "oracion": 4,
        "texto": "Suddenly I became broad awake, and sat up, with a horrible sense of fear upon me, and of some feeling of emptiness around me.",
        "entidades": []
    },
    {
        "oracion": 5,
        "texto": "The room was dark, so I could not see Lucy’s bed; I stole across and felt for her.",
        "entidades": [
            ("Lucy", "PERSON")
        ]
    },
    {
        "oracion": 6,
        "texto": "The bed was empty.",
        "entidades": []
    },
    {
        "oracion": 7,
        "texto": "I lit a match and found that she was not in the room.",
        "entidades": []
    },
    {
        "oracion": 8,
        "texto": "The door was shut, but not locked, as I had left it.",
        "entidades": []
    },
    {
        "oracion": 9,
        "texto": "I feared to wake her mother, who has been more than usually ill lately, so threw on some clothes and got ready to look for her.",
        "entidades": []
    },
    {
        "oracion": 10,
        "texto": "As I was leaving the room it struck me that the clothes she wore might give me some clue to her dreaming intention.",
        "entidades": []
    },
    {
        "oracion": 11,
        "texto": "Dressing-gown would mean house; dress, outside.",
        "entidades": []
    },
    {
        "oracion": 12,
        "texto": "Dressing-gown and dress were both in their places.",
        "entidades": []
    },
    {
        "oracion": 13,
        "texto": "Thank God, I said to myself, she cannot be far, as she is only in her nightdress.",
        "entidades": []
    },
    {
        "oracion": 14,
        "texto": "I ran downstairs and looked in the sitting-room.",
        "entidades": []
    },
    {
        "oracion": 15,
        "texto": "Not there!",
        "entidades": []
    },
    {
        "oracion": 16,
        "texto": "Then I looked in all the other open rooms of the house, with an ever-growing fear chilling my heart.",
        "entidades": []
    },
    {
        "oracion": 17,
        "texto": "Finally I came to the hall door and found it open.",
        "entidades": []
    },
    {
        "oracion": 18,
        "texto": "It was not wide open, but the catch of the lock had not caught.",
        "entidades": []
    },
    {
        "oracion": 19,
        "texto": "The people of the house are careful to lock the door every night, so I feared that Lucy must have gone out as she was.",
        "entidades": [
            ("Lucy", "PERSON")
        ]
    },
    {
        "oracion": 20,
        "texto": "There was no time to think of what might happen; a vague, overmastering fear obscured all details.",
        "entidades": []
    }
]

#mostramos las entidades anotadas manualmente

for anotacion in anotaciones_manuales_dracula:
    print(f"\nOracion {anotacion['oracion']}")
    print(anotacion["texto"])

    if anotacion["entidades"]:
        for span, tipo in anotacion["entidades"]:
            print(f"{span} -> {tipo}")
    else:
        print("Sin entidades")


Oracion 1
I am too agitated to sleep.
Sin entidades

Oracion 2
We have had such an adventure, such an agonising experience.
Sin entidades

Oracion 3
I fell asleep as soon as I had closed my diary.
Sin entidades

Oracion 4
Suddenly I became broad awake, and sat up, with a horrible sense of fear upon me, and of some feeling of emptiness around me.
Sin entidades

Oracion 5
The room was dark, so I could not see Lucy’s bed; I stole across and felt for her.
Lucy -> PERSON

Oracion 6
The bed was empty.
Sin entidades

Oracion 7
I lit a match and found that she was not in the room.
Sin entidades

Oracion 8
The door was shut, but not locked, as I had left it.
Sin entidades

Oracion 9
I feared to wake her mother, who has been more than usually ill lately, so threw on some clothes and got ready to look for her.
Sin entidades

Oracion 10
As I was leaving the room it struck me that the clothes she wore might give me some clue to her dreaming intention.
Sin entidades

Oracion 11
Dressing-gown would 

In [66]:
#evaluamos el ner a nivel de token usando las anotaciones manuales

from sklearn.metrics import precision_recall_fscore_support
import pandas as pd


libros_manuales = {
    "Pride and Prejudice": anotaciones_manuales,
    "Frankenstein": anotaciones_manuales_frankenstein,
    "Dracula": anotaciones_manuales_dracula
}


def construir_bio_manual(texto, entidades):
    doc = nlp(texto)
    etiquetas = ["O"] * len(doc)

    for span, tipo in entidades:
        inicio = texto.find(span)

        if inicio == -1:
            continue

        fin = inicio + len(span)

        tokens_entidad = [
            i for i, token in enumerate(doc)
            if token.idx >= inicio and token.idx + len(token.text) <= fin
        ]

        if tokens_entidad:
            etiquetas[tokens_entidad[0]] = f"B-{tipo}"

            for i in tokens_entidad[1:]:
                etiquetas[i] = f"I-{tipo}"

    return etiquetas


resultados_token = []

for titulo, anotaciones in libros_manuales.items():
    y_true = []
    y_pred = []

    for anotacion in anotaciones:
        texto = anotacion["texto"]
        entidades = anotacion["entidades"]

        doc = nlp(texto)

        etiquetas_true = construir_bio_manual(
            texto,
            entidades
        )

        etiquetas_pred = ["O"] * len(doc)

        for entidad in doc.ents:
            for i, token in enumerate(doc):
                if (
                    token.idx >= entidad.start_char
                    and token.idx + len(token.text) <= entidad.end_char
                ):
                    if token.idx == entidad.start_char:
                        etiquetas_pred[i] = f"B-{entidad.label_}"
                    else:
                        etiquetas_pred[i] = f"I-{entidad.label_}"

        y_true.extend(etiquetas_true)
        y_pred.extend(etiquetas_pred)

    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true,
        y_pred,
        average="micro",
        zero_division=0
    )

    resultados_token.append({
        "libro": titulo,
        "precision": round(precision, 3),
        "recall": round(recall, 3),
        "f1": round(f1, 3)
    })

tabla_token = pd.DataFrame(resultados_token)

display(tabla_token)

,libro,precision,recall,f1
0,Pride and Prejudice,0.950,0.950,0.950
1,Frankenstein,0.962,0.962,0.962
2,Dracula,0.997,0.997,0.997


In [67]:
#evaluamos el ner a nivel de entidad completa
#una entidad solo cuenta como correcta si coincide su span y su tipo

resultados_entidad = []

for titulo, anotaciones in libros_manuales.items():
    verdaderas = []
    predichas = []

    for anotacion in anotaciones:
        texto = anotacion["texto"]

        #entidades manuales con posiciones exactas dentro de la oracion
        for span, tipo in anotacion["entidades"]:
            inicio = texto.find(span)

            if inicio != -1:
                fin = inicio + len(span)

                verdaderas.append(
                    (inicio, fin, tipo)
                )

        #entidades detectadas automaticamente
        doc = nlp(texto)

        for entidad in doc.ents:
            predichas.append(
                (
                    entidad.start_char,
                    entidad.end_char,
                    entidad.label_
                )
            )

    verdaderas = set(verdaderas)
    predichas = set(predichas)

    verdaderos_positivos = len(
        verdaderas.intersection(predichas)
    )

    falsos_positivos = len(
        predichas - verdaderas
    )

    falsos_negativos = len(
        verdaderas - predichas
    )

    precision = (
        verdaderos_positivos /
        (verdaderos_positivos + falsos_positivos)
        if verdaderos_positivos + falsos_positivos > 0
        else 0
    )

    recall = (
        verdaderos_positivos /
        (verdaderos_positivos + falsos_negativos)
        if verdaderos_positivos + falsos_negativos > 0
        else 0
    )

    f1 = (
        2 * precision * recall /
        (precision + recall)
        if precision + recall > 0
        else 0
    )

    resultados_entidad.append({
        "libro": titulo,
        "precision": round(precision, 3),
        "recall": round(recall, 3),
        "f1": round(f1, 3)
    })

tabla_entidad = pd.DataFrame(resultados_entidad)

display(tabla_entidad)

,libro,precision,recall,f1
0,Pride and Prejudice,0.353,0.462,0.400
1,Frankenstein,0.316,0.600,0.414
2,Dracula,0.667,1.000,0.800


In [68]:
#unimos los resultados de las dos formas de evaluacion en una sola tabla

tabla_comparativa = pd.concat([
    tabla_token.assign(evaluacion="Nivel de token"),
    tabla_entidad.assign(evaluacion="Entidad completa")
], ignore_index=True)

tabla_comparativa = tabla_comparativa[
    ["libro", "evaluacion", "precision", "recall", "f1"]
]

display(tabla_comparativa)

,libro,evaluacion,precision,recall,f1
0,Pride and Prejudice,Nivel de token,0.950,0.950,0.950
1,Frankenstein,Nivel de token,0.962,0.962,0.962
2,Dracula,Nivel de token,0.997,0.997,0.997
3,Pride and Prejudice,Entidad completa,0.353,0.462,0.400
4,Frankenstein,Entidad completa,0.316,0.600,0.414
5,Dracula,Entidad completa,0.667,1.000,0.800


La evaluacion por tokens si es bastante mas optimista en los tres libros. Los f1 score son 0.95, 0.96 y 0.99 respectivamente. A nivel de entidad completa bajan a 0.40, 0.41 y 0.80 tambien. Entre los libros, Dracula si claramente presenta el mejor resultado en ambas evaluaciones, especialmente a nivel de entidad completa, mientras que pride and prejudice y frankestein si tienen resultados mucho mas bajos en esa metrica.

**PREGUNTAS**

In [69]:
#calculamos el porcentaje de tokens que aparecen dentro de dialogos

resultados_dialogo = []

for _, fila in novelas.iterrows():
    titulo = fila["title"]
    texto = str(fila["text"])

    #buscamos texto entre comillas dobles o tipograficas
    dialogos = re.findall(
        r'["“](.*?)["”]',
        texto,
        flags=re.DOTALL
    )

    tokens_totales = len(patron_palabras.findall(texto))

    tokens_dialogo = sum(
        len(patron_palabras.findall(dialogo))
        for dialogo in dialogos
    )

    porcentaje_dialogo = (
        tokens_dialogo / tokens_totales * 100
        if tokens_totales > 0
        else 0
    )

    resultados_dialogo.append({
        "libro": titulo,
        "tokens_totales": tokens_totales,
        "tokens_dialogo": tokens_dialogo,
        "porcentaje_dialogo": round(porcentaje_dialogo, 2)
    })

tabla_dialogo = pd.DataFrame(resultados_dialogo)

display(tabla_dialogo)

,libro,tokens_totales,tokens_dialogo,porcentaje_dialogo
0,Pride and Prejudice,128366,58039,45.21
1,Frankenstein,75303,30422,40.40
2,Dracula,162771,58469,35.92


En la distribucion POS Frankestein si tiene mas ADJ y NOUN que los otros dos, con 160 ADJ y 452 NOUN respectivamente. Pride and Prejudice tambien tienen mas PROPN con 97 frente a 37 de Frankestein y unos 60 en Dracula. Tambien se tiene 110 entidades PERSON frente a 41 y 77. En lo que al dialogo respecta, el porcentaje de tokens dentro de los dialogos es 45.21 en Pride and Prejudice, 40.4 en Frankestein y 35.92 en Dracula. En estos resultados, pride and prejudice presentan la mayor precencia de dialogo y de nombres propios tambien.

Ambos problemas dependen del contexto. En POS, una misma palabra puede recibir distintas etiquetas según las palabras que la rodean. En un modelo n-grama, la siguiente palabra tambien se predice condicionada por las palabras anteriores. Por eso, en ambos casos la informacion del contexto es fundamental.

En POS, la diferencia de dominio puede explicar errores con palabras o construcciones menos comunes, como Bennet -> NN, quite -> PDT o la separacion de Good-bye. En NER, puede dificultar el reconocimiento correcto de nombres y lugares poco familiares para el modelo. Esto se relaciona directamente con el OOV del Laboratorio 5, alli se encontro que 1,514 palabras del conjunto de prueba no habian aparecido en entrenamiento, y se uso UNK para poder manejarlas. En textos literarios, nombres propios, formas antiguas y vocabulario poco frecuente pueden producir un problema similar.

Personalmente usaria F1 de entidad completa. En el notebook, el F1 por token fue 0.950, 0.962 y 0.997, mientras que por entidad completa fue 0.400, 0.414 y 0.800. La evaluacion completa es mas estricta porque exige que coincidan el span y el tipo de la entidad.

En conjunto, los resultados muestran que las tres novelas no producen exactamente el mismo comportamiento en las herramientas de NLP. Hay diferencias en la distribución de categorias gramaticales, en la cantidad y tipos de entidades y en la proporción de dialogo. Ademas, el desempeno de NER cambia bastante según la forma de evaluacion. Esto muestra que las caracteristicas del texto influyen en cómo responden modelos entrenados con datos contemporaneos.